# 01_FE
---

![](img/fe.png)

In [1]:
# se limpia memoria
rm(list=ls())
gc()

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,650807,34.8,1447464,77.4,NA,1071068,57.3
Vcells,1211957,9.3,8388608,64.0,16384,1964451,15.0


In [3]:
library(yaml)
require("data.table")
require("Rcpp")
require("rlist")
require("yaml")
library(dplyr)
require("lightgbm")
require("ranger")
require("randomForest")  #solo se usa para imputar nulos

In [4]:
setwd("/Users/ppando/Materias/cs-datos/especializacion/ciencia-datos/exp/")

In [5]:
PARAMS <- yaml.load_file("01_FE_prxs_01a.yml")

In [6]:
ReportarCampos <- function(dataset) {
  cat("La cantidad de campos es ", ncol(dataset), "\n")
}

In [7]:
# se agrega al dataset una variable que va de 1 a 12, el mes, para que el modelo aprenda estacionalidad
AgregarMes <- function(dataset) {
  gc()
  dataset[ , mes := foto_mes %% 100]
  ReportarCampos(dataset)
}

In [8]:
# se eliminan las variables que uno supone hace Data Drifting
DriftEliminar <- function(dataset, variables) {
  gc()
  dataset[ , c(variables) := NULL]
  ReportarCampos(dataset)
}

In [9]:
# a las variables que tienen nulos, se les agrega una nueva variable el dummy de is es nulo o no {0, 1}
DummiesNA  <- function(dataset) {
  gc()
  nulos <- colSums(is.na(dataset[ foto_mes %in% PARAMS$param$const$futuro]))  # cuenta la cantidad de nulos por columna
  colsconNA <- names(which(nulos > 0))

  dataset[ , paste0(colsconNA, "_isNA") := lapply(.SD, is.na), .SDcols= colsconNA]

  ReportarCampos(dataset)
}

In [10]:
# se corrige poniendo a NA las variables que en ese mes estan dañadas
CorregirNA <- function(dataset) {
  gc()
  #acomodo los errores del dataset
  # dataset[ foto_mes==201801,  internet   := NA ]
  # dataset[ foto_mes==201801,  thomebanking   := NA ]
  ReportarCampos(dataset)
}

In [11]:
AgregarVariables <- function(dataset) {
  gc()
  #INICIO de la seccion donde se deben hacer cambios con variables nuevas

  #Aqui debe usted agregar sus propias nuevas variables

  #valvula de seguridad para evitar valores infinitos
  #paso los infinitos a NULOS
  infinitos      <- lapply(names(dataset), function(.name) dataset[ , sum(is.infinite(get(.name)))])
  infinitos_qty  <- sum(unlist(infinitos))
  if(infinitos_qty > 0) {
    cat("ATENCION, hay", infinitos_qty, "valores infinitos en tu dataset. Seran pasados a NA\n")
    dataset[mapply(is.infinite, dataset)] <<- NA
  }

  #valvula de seguridad para evitar valores NaN  que es 0/0
  #paso los NaN a 0 , decision polemica si las hay
  #se invita a asignar un valor razonable segun la semantica del campo creado
  nans      <- lapply(names(dataset),function(.name) dataset[ , sum(is.nan(get(.name)))])
  nans_qty  <- sum(unlist(nans))
  if( nans_qty > 0 ) {
    cat("ATENCION, hay", nans_qty, "valores NaN 0/0 en tu dataset. Seran pasados arbitrariamente a 0\n")
    cat("Si no te gusta la decision, modifica a gusto el programa!\n\n")
    dataset[mapply(is.nan, dataset)] <<- 0
  }

  ReportarCampos(dataset)
}

In [12]:
# esta funcion supone que dataset esta ordenado por <medico, foto_mes>
# se calcula el lag y el delta lag

Lags  <- function(cols, nlag, deltas) {
  gc()
  sufijo  <- paste0("_lag", nlag)

  dataset[ , paste0(cols, sufijo) := shift(.SD, nlag, NA, "lag"), 
             by= medico, 
             .SDcols= cols]

  # agrego los deltas de los lags, con un "for" nada elegante
  if(deltas) {
    sufijodelta  <- paste0("_delta", nlag)

    for(vcol in cols) {
     dataset[,  paste0(vcol, sufijodelta) := get(vcol)  - get(paste0(vcol, sufijo))]
    }
  }

  ReportarCampos(dataset)
}

In [13]:
# se calculan para los 6 meses previos el minimo, maximo y tendencia calculada con cuadrados minimos
# la formula de calculo de la tendencia puede verse en https://stats.libretexts.org/Bookshelves/Introductory_Statistics/Book%3A_Introductory_Statistics_(Shafer_and_Zhang)/10%3A_Correlation_and_Regression/10.04%3A_The_Least_Squares_Regression_Line
# para la maxíma velocidad esta funcion esta escrita en lenguaje C, y no en la porqueria de R o Python

cppFunction('NumericVector fhistC(NumericVector pcolumna, IntegerVector pdesde ) 
{
  /* Aqui se cargan los valores para la regresion */
  double  x[100] ;
  double  y[100] ;

  int n = pcolumna.size();
  NumericVector out( 5*n );

  for(int i = 0; i < n; i++)
  {
    //lag
    if( pdesde[i]-1 < i )  out[ i + 4*n ]  =  pcolumna[i-1] ;
    else                   out[ i + 4*n ]  =  NA_REAL ;


    int  libre    = 0 ;
    int  xvalor   = 1 ;

    for( int j= pdesde[i]-1;  j<=i; j++ )
    {
       double a = pcolumna[j] ;

       if( !R_IsNA( a ) ) 
       {
          y[ libre ]= a ;
          x[ libre ]= xvalor ;
          libre++ ;
       }

       xvalor++ ;
    }

    /* Si hay al menos dos valores */
    if( libre > 1 )
    {
      double  xsum  = x[0] ;
      double  ysum  = y[0] ;
      double  xysum = xsum * ysum ;
      double  xxsum = xsum * xsum ;
      double  vmin  = y[0] ;
      double  vmax  = y[0] ;

      for( int h=1; h<libre; h++)
      { 
        xsum  += x[h] ;
        ysum  += y[h] ; 
        xysum += x[h]*y[h] ;
        xxsum += x[h]*x[h] ;

        if( y[h] < vmin )  vmin = y[h] ;
        if( y[h] > vmax )  vmax = y[h] ;
      }

      out[ i ]  =  (libre*xysum - xsum*ysum)/(libre*xxsum -xsum*xsum) ;
      out[ i + n ]    =  vmin ;
      out[ i + 2*n ]  =  vmax ;
      out[ i + 3*n ]  =  ysum / libre ;
    }
    else
    {
      out[ i       ]  =  NA_REAL ; 
      out[ i + n   ]  =  NA_REAL ;
      out[ i + 2*n ]  =  NA_REAL ;
      out[ i + 3*n ]  =  NA_REAL ;
    }
  }

  return  out;
}')

In [14]:
# se calcula la tendencia de las variables cols de los ultimos 6 meses
# la tendencia es la pendiente de la recta que ajusta por cuadrados minimos

TendenciaYmuchomas  <- function(dataset, cols, ventana=6, tendencia=TRUE, minimo=TRUE, maximo=TRUE, promedio=TRUE, 
                                ratioavg=FALSE, ratiomax=FALSE) {
  gc()
  # cantidad de meses que utilizo para la historia
  ventana_regresion  <- ventana

  last  <- nrow( dataset )

  # creo el vector_desde que indica cada ventana
  # de esta forma se acelera el procesamiento ya que lo hago una sola vez
  vector_ids   <- dataset$medico

  vector_desde  <- seq(-ventana_regresion+2,  nrow(dataset)-ventana_regresion+1)
  vector_desde[1:ventana_regresion]  <-  1

  for (i in 2:last)
      if (vector_ids[i-1] != vector_ids[i]) {
          vector_desde[i] <-  i
      }
  for (i in 2:last)
      if (vector_desde[i] < vector_desde[i-1]) {
          vector_desde[i] <-  vector_desde[i-1]
      }

  for (campo in cols) {
    nueva_col <- fhistC( dataset[ , get(campo)], vector_desde) 

    if (tendencia)  dataset[ , paste0( campo, "_tend", ventana) := nueva_col[ (0*last +1):(1*last) ]  ]
    if (minimo)     dataset[ , paste0( campo, "_min", ventana)  := nueva_col[ (1*last +1):(2*last) ]  ]
    if (maximo)     dataset[ , paste0( campo, "_max", ventana)  := nueva_col[ (2*last +1):(3*last) ]  ]
    if (promedio)   dataset[ , paste0( campo, "_avg", ventana)  := nueva_col[ (3*last +1):(4*last) ]  ]
    if (ratioavg)   dataset[ , paste0( campo, "_ratioavg", ventana)  := get(campo) /nueva_col[ (3*last +1):(4*last) ]  ]
    if (ratiomax)   dataset[ , paste0( campo, "_ratiomax", ventana)  := get(campo) /nueva_col[ (2*last +1):(3*last) ]  ]
  }

   ReportarCampos(dataset)
}

In [15]:
# elimina del dataset las variables que estan por debajo de la capa geologica de canaritos
# se llama varias veces, luego de agregar muchas variables nuevas, para ir reduciendo la cantidad de variables
# y así hacer lugar a nuevas variables importantes

GVEZ <- 1 

CanaritosImportancia <- function(canaritos_ratio=0.2) {
  gc()
  ReportarCampos(dataset)

  for (i  in 1:(ncol(dataset)*canaritos_ratio))  
      dataset[ , paste0("canarito", i) :=  runif(nrow(dataset))]
  campos_buenos  <- setdiff(colnames(dataset), c("medico","foto_mes","clase"))
  azar  <- runif(nrow(dataset))
  dataset[ , entrenamiento := foto_mes>= 202001 &  foto_mes< 202106 & azar < 0.10]
  
  dtrain  <- lgb.Dataset( data=    data.matrix(  dataset[ entrenamiento==TRUE, campos_buenos, with=FALSE]),
                          label=   dataset[ entrenamiento==TRUE, clase],
                          free_raw_data= FALSE
  )
  
  dvalid  <- lgb.Dataset( data=    data.matrix(  dataset[ foto_mes==202106, campos_buenos, with=FALSE]),
                          label=   dataset[ foto_mes==202106, clase],
                          free_raw_data= FALSE
  )
  
  param <- list( objective= "regression",
                 metric= "rmse",
                 first_metric_only= TRUE,
                 boost_from_average= TRUE,
                 feature_pre_filter= FALSE,
                 verbosity= -100,
                 seed= 999983,
                 max_depth=  -1,         # -1 significa no limitar,  por ahora lo dejo fijo
                 min_gain_to_split= 0.0, # por ahora, lo dejo fijo
                 lambda_l1= 0.0,         # por ahora, lo dejo fijo
                 lambda_l2= 0.0,         # por ahora, lo dejo fijo
                 max_bin= 31,            # por ahora, lo dejo fijo
                 num_iterations= 9999,   # un numero muy grande, lo limita early_stopping_rounds
                 force_row_wise= TRUE,   # para que los alumnos no se atemoricen con tantos warning
                 learning_rate= 0.065, 
                 feature_fraction= 1.0,  # lo seteo en 1 para que las primeras variables del dataset no se vean opacadas
                 min_data_in_leaf= 260,
                 num_leaves= 60,
                 # num_threads= 8,
                 early_stopping_rounds= 200 )
  
  modelo  <- lgb.train( data= dtrain,
                        valids= list( valid= dvalid ),
                        # eval= fganancia_lgbm_meseta,
                        param= param,
                        verbose= -100 )
  
  tb_importancia  <- lgb.importance(model= modelo)
  tb_importancia[  , pos := .I ]
  
  GVEZ  <<- GVEZ + 1
  
  umbral  <- tb_importancia[Feature %like% "canarito", median(pos) + 2*sd(pos)] 
  
  col_utiles  <- tb_importancia[ pos < umbral & !( Feature %like% "canarito"),  Feature ]
  col_utiles  <-  unique( c( col_utiles,  c("medico","foto_mes","clase","mes") ) )
  col_inutiles  <- setdiff( colnames(dataset), col_utiles )
  
  dataset[ ,  (col_inutiles) := NULL]
  
  ReportarCampos(dataset)
}

In [16]:
# agrega para caca columna de cols una nueva variable _rank  que es un numero entre 0 y 1  del ranking de esa variable ese mes
Rankeador <- function(cols) {
  gc()
  sufijo  <- "_rank" 

  for (vcol in cols) {
     dataset[ , paste0(vcol, sufijo) := frank( get(vcol), ties.method= "random")/ .N, by= foto_mes]
  }

  ReportarCampos(dataset)
}

## Main

In [17]:
# se carga el dataset

nom_arch <- PARAMS$param$files$input$dentrada 
dataset  <- readRDS(nom_arch)

dataset[, c("producto","row"):=NULL]

dataset <- dataset[, .(prescripciones=sum(pxs,na.rm = T)),by=c("medico", "esp1", "foto_mes", "labn")]


dataset <- dcast(dataset,medico+esp1+foto_mes~labn,fun.aggregate = sum,value.var = "prescripciones")
setnames(dataset, PARAMS$param$const$lab_clase, PARAMS$param$const$clase)

In [18]:
# se ordena el dataset por <medico, foto_mes> para poder hacer lags
setorderv(dataset, PARAMS$param$const$campos_sort)
# dataset[medico=="zx1xv",.(medico,foto_mes,clase,rollclase=frollmean(clase,6,align = "left",na.rm = T))][medico=="zx1xv",.(medico,foto_mes,clase,rollclase,leadedclase=lead(rollclase))] %>% view()

In [19]:
# se crea la clase final

dataset[, rolledclase:=frollmean(clase,6,align = "left",na.rm = T),by=c("medico")]
dataset[, rolledclase:=lead(rolledclase),by=c("medico")]
# dataset <- dataset[!is.na(rolledclase)]
dataset[, clase:=NULL]
setnames(dataset, "rolledclase" ,"clase")

# dataset[medico=="zx1xv"] %>% View()

AgregarMes(dataset)  #agrego el mes del año

if (length(PARAMS$param$variablesdrift) > 0)
    DriftEliminar(dataset, PARAMS$param$variablesdrift)

if (PARAMS$param$dummiesNA)
    DummiesNA(dataset) 

if (PARAMS$param$corregir == "AsignarNA")
    CorregirNA(dataset) 

if (PARAMS$param$variablesmanuales)
    AgregarVariables(dataset)

cols_lagueables  <- copy(setdiff(colnames(dataset), PARAMS$param$const$campos_fijos))

for (i in 1:length( PARAMS$param$tendenciaYmuchomas$correr)) {
  if (PARAMS$param$tendenciaYmuchomas$correr[i]) {
    #veo si tengo que ir agregando variables
    if (PARAMS$param$acumulavars)  
        cols_lagueables  <- setdiff(colnames(dataset), PARAMS$param$const$campos_fijos)

    cols_lagueables  <- intersect(colnames(dataset), cols_lagueables)
    TendenciaYmuchomas( dataset, 
                        cols     = cols_lagueables,
                        ventana  = PARAMS$param$tendenciaYmuchomas$ventana[i],
                        tendencia= PARAMS$param$tendenciaYmuchomas$tendencia[i],
                        minimo=    PARAMS$param$tendenciaYmuchomas$minimo[i],
                        maximo=    PARAMS$param$tendenciaYmuchomas$maximo[i],
                        promedio=  PARAMS$param$tendenciaYmuchomas$promedio[i],
                        ratioavg=  PARAMS$param$tendenciaYmuchomas$ratioavg[i],
                        ratiomax=  PARAMS$param$tendenciaYmuchomas$ratiomax[i]
                      )
  # elimino las variables poco importantes, para hacer lugar a las importantes
    if (PARAMS$param$tendenciaYmuchomas$canaritos[ i ] > 0)  
        CanaritosImportancia(canaritos_ratio= unlist(PARAMS$param$tendenciaYmuchomas$canaritos[ i ]))

  }
}


for (i in 1:length(PARAMS$param$lags$correr)) {
  if (PARAMS$param$lags$correr[i]) {
    #veo si tengo que ir agregando variables
    if (PARAMS$param$acumulavars)  
        cols_lagueables <- setdiff(colnames(dataset), PARAMS$param$const$campos_fijos)

    cols_lagueables  <- intersect(colnames(dataset), cols_lagueables)
    Lags( cols_lagueables, 
          PARAMS$param$lags$lag[i], 
          PARAMS$param$lags$delta[i])   # calculo los lags de orden  i

    #elimino las variables poco importantes, para hacer lugar a las importantes
    if (PARAMS$param$lags$canaritos[ i ] > 0)
        CanaritosImportancia( canaritos_ratio= unlist(PARAMS$param$lags$canaritos[ i ]))
  }
}


if (PARAMS$param$acumulavars)  
    cols_lagueables  <- setdiff(colnames(dataset), PARAMS$param$const$campos_fijos)

if (PARAMS$param$rankeador) {
  if (PARAMS$param$acumulavars)  
      cols_lagueables  <- setdiff(colnames(dataset), PARAMS$param$const$campos_fijos)

  cols_lagueables  <- intersect(colnames(dataset), cols_lagueables)
  setorderv(dataset, PARAMS$param$const$campos_rsort)
  Rankeador(cols_lagueables)
  setorderv(dataset, PARAMS$param$const$campos_sort)
}

if (PARAMS$param$canaritos_final > 0)
    CanaritosImportancia(canaritos_ratio= PARAMS$param$canaritos_final)

# se deja la clase como ultimo campo
nuevo_orden  <- c(setdiff(colnames(dataset) , PARAMS$param$const$clase) , PARAMS$param$const$clase)
setcolorder(dataset, nuevo_orden)

# se corrije los NANs que surgieron de los ratios
cols_lagueables  <- copy(setdiff(colnames(dataset), PARAMS$param$const$campos_fijos))
for (col in cols_lagueables) {
  dataset[[col]][is.nan(dataset[[col]])] <- 0
}

La cantidad de campos es  13 
La cantidad de campos es  61 
La cantidad de campos es  61 
La cantidad de campos es  25 
La cantidad de campos es  43 
La cantidad de campos es  43 
La cantidad de campos es  29 
La cantidad de campos es  39 
La cantidad de campos es  39 
La cantidad de campos es  35 
La cantidad de campos es  39 
La cantidad de campos es  39 
La cantidad de campos es  38 
La cantidad de campos es  42 
La cantidad de campos es  42 
La cantidad de campos es  35 
La cantidad de campos es  35 
La cantidad de campos es  33 


In [20]:
# se graba el dataset
fwrite(dataset, PARAMS$param$files$output$dsalida, logical01= TRUE, sep= ",")